# Train Embeddings Clustering

This notebook:
- Loads `Data/train/train_data_chunks.csv`
- Extracts embedding columns and converts to a NumPy array
- Uses column names as feature names
- Runs the feature clustering pipeline from `src/clustering.py`
- Targets 40–60 clusters with minimum cluster size of 3


In [1]:
# Setup: imports and repo root so we can import src/clustering
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Ensure project root is on sys.path
this_nb = Path.cwd()
# If running from notebooks/, project root is parent
project_root = this_nb if (this_nb / 'src').exists() else this_nb.parent
if not (project_root / 'src').exists():
    # fallback: ascend until src exists (max 3 levels)
    cur = this_nb
    for _ in range(3):
        if (cur / 'src').exists():
            project_root = cur
            break
        cur = cur.parent

sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

from src.clustering import run_clustering_pipeline


Project root: /Users/taishajoseph/Documents/Projects/MDC-Challenge-2025


2025-09-05 21:01:27,240 - src.helpers - INFO - Logging initialized for /Users/taishajoseph/Documents/Projects/MDC-Challenge-2025/logs/duckdb_utils.log
2025-09-05 21:01:27,240 - src.helpers - INFO - Logging initialized for /Users/taishajoseph/Documents/Projects/MDC-Challenge-2025/logs/duckdb_utils.log


In [2]:
# Import per-cluster PCA helper
from src.dimensionality_reduction import Reducer


In [3]:
# Load CSV and identify embedding columns
csv_path = project_root / 'Data' / 'train' / 'train_data_chunks.csv'
print(f"Loading: {csv_path}")

df_head = pd.read_csv(csv_path, nrows=5)
all_cols = list(df_head.columns)

# Heuristics: embedding columns are float-like and often named like 'emb_*' or numeric index columns
# We'll read a small chunk to infer dtypes, then select all float columns as embeddings.
float_cols = [c for c, dtype in df_head.infer_objects().dtypes.items() if np.issubdtype(dtype, np.number)]

# If no float columns found from head, fall back to full dtype inference with low_memory=False
if not float_cols:
    df_probe = pd.read_csv(csv_path, nrows=1000)
    float_cols = [c for c, dtype in df_probe.dtypes.items() if np.issubdtype(dtype, np.number)]

# Optionally filter by name pattern if present
name_patterns = ('emb', 'embedding', 'feature_', 'f_')
embed_cols = [c for c in float_cols if any(p in c.lower() for p in name_patterns)] or float_cols

print(f"Detected {len(embed_cols)} embedding columns")
embed_cols[:10]


Loading: /Users/taishajoseph/Documents/Projects/MDC-Challenge-2025/Data/train/train_data_chunks.csv
Detected 384 embedding columns


['emb_0',
 'emb_1',
 'emb_2',
 'emb_3',
 'emb_4',
 'emb_5',
 'emb_6',
 'emb_7',
 'emb_8',
 'emb_9']

In [4]:
# Build NumPy array and feature names
usecols = embed_cols

df = pd.read_csv(csv_path) # get chunk IDs from first column
if 'chunk_id' not in df.columns:
    raise KeyError("Expected first column named 'chunk_id' not found in CSV")
chunk_ids = df['chunk_id'].astype(str).tolist()
len(chunk_ids), chunk_ids[:3]

(7736,
 ['10.1093_beheco_arz101_0',
  '10.1093_beheco_arz101_1',
  '10.1093_beheco_arz101_2'])

In [5]:
df

,chunk_id,document_id,token_count,target,target_str,emb_0,emb_1,emb_2,emb_3,emb_4,...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,10.1093_beheco_arz101_0,10.1093_beheco_arz101,495,0,NOT RELEVANT,0.015713,-0.009378,0.020453,0.024954,0.017898,...,-0.015458,0.023589,-0.032979,-0.037276,-0.005317,0.054605,0.026833,-0.019565,0.098890,0.055512
1,10.1093_beheco_arz101_1,10.1093_beheco_arz101,403,0,NOT RELEVANT,-0.015680,0.006466,0.015703,0.030072,0.044212,...,-0.005484,0.024652,0.004320,-0.028463,0.016289,0.062229,0.005102,-0.029444,0.099293,0.035085
2,10.1093_beheco_arz101_2,10.1093_beheco_arz101,477,0,NOT RELEVANT,-0.007936,-0.007952,-0.004148,0.034437,0.008208,...,-0.021554,0.055830,-0.010331,0.030738,0.052802,0.080657,0.007860,-0.060096,0.083487,0.047805
3,10.1093_beheco_arz101_3,10.1093_beheco_arz101,490,0,NOT RELEVANT,0.014775,-0.014930,0.028272,0.025603,0.000877,...,-0.008698,0.036334,0.005779,-0.037128,0.034317,0.100105,-0.007818,-0.001108,0.115332,0.107688
4,10.1093_beheco_arz101_4,10.1093_beheco_arz101,454,0,NOT RELEVANT,-0.000533,-0.041749,0.001220,0.048582,-0.003726,...,-0.033520,0.051418,0.043521,-0.032925,0.055606,0.084604,-0.000608,-0.036608,0.108437,0.083489
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7731,10.1145_3461702.3462538_51,10.1145_3461702.3462538,877,0,NOT RELEVANT,0.015056,-0.029600,-0.029355,-0.012678,0.056834,...,0.029724,-0.033095,0.035024,0.038820,-0.018248,-0.036209,0.006853,-0.016966,0.048283,0.001238
7732,10.1145_3461702.3462538_52,10.1145_3461702.3462538,800,0,NOT RELEVANT,-0.015578,-0.021899,0.006349,-0.022127,0.072598,...,0.013217,-0.057269,0.003375,0.050494,-0.024504,-0.043790,0.017158,-0.014025,0.025087,0.001920
7733,10.1145_3461702.3462538_53,10.1145_3461702.3462538,552,0,NOT RELEVANT,-0.038412,-0.003948,-0.059742,0.039143,0.041381,...,0.023371,-0.042475,-0.038102,0.031477,0.010990,-0.026281,0.020221,-0.062554,0.003291,0.016057
7734,10.1145_3461702.3462538_54,10.1145_3461702.3462538,798,0,NOT RELEVANT,-0.018164,0.011153,-0.009743,0.026075,0.060471,...,0.060976,-0.039634,-0.006364,0.021465,-0.023340,-0.042957,0.027916,-0.016218,0.036270,-0.006396


In [6]:
# keep only embedding columns
feature_names = list(df[usecols].columns)
X = df[usecols].to_numpy(dtype=np.float32, copy=False)
print(X.shape)
feature_names[:10]

(7736, 384)


['emb_0',
 'emb_1',
 'emb_2',
 'emb_3',
 'emb_4',
 'emb_5',
 'emb_6',
 'emb_7',
 'emb_8',
 'emb_9']

In [7]:
# Run clustering pipeline aiming for 40–60 clusters, min cluster size = 3
# We'll set target_n=50 with tol=10 to allow 40–60, and keep min_cluster_size=3

feature_cluster_map = run_clustering_pipeline(
    dataset_embeddings=X,
    feature_names=feature_names,
    k_neighbors=3,
    similarity_threshold=None,
    threshold_method='degree_target',
    target_n=48,
    tol=10,
    min_cluster_size=3,
    max_cluster_size=9999,
    split_factor=1.3,
    random_seed=42,
    output_dir=str(project_root / 'reports' / 'clustering')
)

len(set(feature_cluster_map.values()))


Executing run_clustering_pipeline...
Executing build_knn_similarity_graph...
Executing determine_similarity_threshold...
Function determine_similarity_threshold took 0.0522 seconds to complete.
Function build_knn_similarity_graph took 0.1569 seconds to complete.
Executing find_resolution_for_target...
Executing run_leiden_clustering...
Function run_leiden_clustering took 0.0203 seconds to complete.
Function find_resolution_for_target took 0.0206 seconds to complete.
Executing export_feature_clusters...
Function export_feature_clusters took 0.0375 seconds to complete.
Executing export_clustering_report...
Function export_clustering_report took 0.0022 seconds to complete.
Function run_clustering_pipeline took 0.2178 seconds to complete.


41

In [8]:
# Preview and save outputs
from collections import Counter

n_clusters = len(set(feature_cluster_map.values()))
print(f"Clusters found: {n_clusters}")

cluster_counts = Counter(feature_cluster_map.values())
print('Smallest cluster size:', min(cluster_counts.values()))
print('Largest cluster size:', max(cluster_counts.values()))

# Save a CSV mapping feature -> cluster
out_dir = project_root / 'reports' / 'clustering'
out_dir.mkdir(parents=True, exist_ok=True)
map_df = pd.DataFrame({'feature_name': list(feature_cluster_map.keys()),
                       'cluster': list(feature_cluster_map.values())})
map_path = out_dir / 'feature_clusters.csv'
map_df.to_csv(map_path, index=False)
map_path


Clusters found: 41
Smallest cluster size: 3
Largest cluster size: 16


PosixPath('/Users/taishajoseph/Documents/Projects/MDC-Challenge-2025/reports/clustering/feature_clusters.csv')

## Per-cluster PCA

In [ ]:
# Per-cluster PCA (no DB writes) and save reduced CSV
from collections import defaultdict
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

# Ensure we have a feature_cluster_map available
if 'feature_cluster_map' not in globals():
    map_file = project_root / 'reports' / 'clustering' / 'feature_clusters.json'
    if map_file.exists():
        with open(map_file, 'r') as f:
            feature_cluster_map = json.load(f)
        print(f"Loaded feature_cluster_map from {map_file}")

# Map features to column indices for each cluster
name_to_idx = {fname: i for i, fname in enumerate(feature_names)}
cluster2cols = defaultdict(list)
missing_features = 0
for fname, cid in feature_cluster_map.items():
    idx = name_to_idx.get(fname)
    if idx is not None:
        cluster2cols[cid].append(idx)
    else:
        missing_features += 1
if missing_features:
    print(f"Warning: {missing_features} features in cluster map not found in current matrix; ignored.")

# Compute PC1 per cluster using the Reducer helper
reducer = Reducer()

assert len(chunk_ids) == X.shape[0], "Mismatch between number of chunk_ids and rows in embeddings matrix"

pc1_results = []  # list of (cluster_id, pc1_vector)
with ThreadPoolExecutor(max_workers=min(8, max(1, len(cluster2cols)))) as ex:
    futures = {
        ex.submit(reducer._run_pca_on_cluster, cid, cols, X, 42): cid
        for cid, cols in cluster2cols.items()
        if len(cols) >= 1
    }
    for fut in as_completed(futures):
        cid = futures[fut]
        pc1 = fut.result()
        pc1_results.append((cid, pc1))

if not pc1_results:
    raise RuntimeError("No per-cluster PCA results produced.")

# Assemble reduced matrix and DataFrame
pc1_results.sort()  # stable order by cluster id
labels, vectors = zip(*pc1_results)
X_reduced = np.column_stack(vectors)  # shape: (n_samples, n_clusters)

reduced_df = pd.DataFrame({'chunk_id': chunk_ids})
for j, cid in enumerate(labels):
    reduced_df[f"LEIDEN_{cid}"] = X_reduced[:, j].astype(float)

2025-09-05 21:01:29,692 - api.database.duckdb_schema - INFO - Starting DuckDB schema creation...
2025-09-05 21:01:29,692 - api.database.duckdb_schema - INFO - Starting DuckDB schema creation...
2025-09-05 21:01:29,692 - api.database.duckdb_schema - INFO - Starting DuckDB schema creation...
2025-09-05 21:01:29,692 - api.database.duckdb_schema - INFO - Starting DuckDB schema creation...
2025-09-05 21:01:29,700 - api.database.duckdb_schema - INFO - Creating documents table...
2025-09-05 21:01:29,700 - api.database.duckdb_schema - INFO - Creating documents table...
2025-09-05 21:01:29,700 - api.database.duckdb_schema - INFO - Creating documents table...
2025-09-05 21:01:29,700 - api.database.duckdb_schema - INFO - Creating documents table...
2025-09-05 21:01:29,705 - api.database.duckdb_schema - INFO - Documents table created successfully
2025-09-05 21:01:29,705 - api.database.duckdb_schema - INFO - Documents table created successfully
2025-09-05 21:01:29,705 - api.database.duckdb_schema -

In [ ]:
# merge back with original df
merged_df = pd.merge(df.drop(columns=usecols), reduced_df, on='chunk_id', how='left')
merged_df

,chunk_id,document_id,token_count,target,target_str,LEIDEN_cluster_0,LEIDEN_cluster_1,LEIDEN_cluster_10,LEIDEN_cluster_11,LEIDEN_cluster_12,...,LEIDEN_cluster_37,LEIDEN_cluster_38,LEIDEN_cluster_39,LEIDEN_cluster_4,LEIDEN_cluster_40,LEIDEN_cluster_5,LEIDEN_cluster_6,LEIDEN_cluster_7,LEIDEN_cluster_8,LEIDEN_cluster_9
0,10.1093_beheco_arz101_0,10.1093_beheco_arz101,495,0,NOT RELEVANT,0.019341,0.071276,0.013338,0.017317,0.100030,...,0.071350,0.025864,0.049635,0.024202,-0.029292,-0.013984,0.011096,0.007099,-0.051095,0.043329
1,10.1093_beheco_arz101_1,10.1093_beheco_arz101,403,0,NOT RELEVANT,0.039789,0.049795,-0.009494,0.047296,0.045410,...,0.086206,0.032228,0.037546,0.048711,-0.030125,-0.021250,-0.011081,-0.033776,-0.030804,0.005051
2,10.1093_beheco_arz101_2,10.1093_beheco_arz101,477,0,NOT RELEVANT,0.074431,0.053482,0.003031,0.026165,0.012989,...,0.071322,0.044600,0.045440,-0.004575,0.030543,0.015371,-0.020847,-0.055483,-0.016575,-0.001612
3,10.1093_beheco_arz101_3,10.1093_beheco_arz101,490,0,NOT RELEVANT,0.054360,0.092126,0.038252,0.007297,0.028315,...,0.034088,0.005983,0.039476,0.036475,-0.035952,-0.015191,-0.002271,0.000679,-0.034913,-0.008656
4,10.1093_beheco_arz101_4,10.1093_beheco_arz101,454,0,NOT RELEVANT,0.078152,0.079728,-0.024161,0.027902,0.023583,...,0.074596,0.001763,0.014274,-0.004581,0.023368,0.010099,-0.011894,0.006178,-0.019626,-0.039457
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7731,10.1145_3461702.3462538_51,10.1145_3461702.3462538,877,0,NOT RELEVANT,0.049745,-0.091395,-0.042592,-0.079896,0.052467,...,0.004027,-0.038853,-0.001158,0.041306,-0.072631,0.095917,0.001894,0.048311,0.034365,0.038191
7732,10.1145_3461702.3462538_52,10.1145_3461702.3462538,800,0,NOT RELEVANT,0.055301,-0.103066,-0.070817,-0.045065,-0.019605,...,0.006807,-0.016219,-0.039629,0.128129,-0.068540,0.131169,0.033777,0.087518,0.055862,-0.026203
7733,10.1145_3461702.3462538_53,10.1145_3461702.3462538,552,0,NOT RELEVANT,0.048941,-0.080665,-0.007966,-0.049330,0.025307,...,-0.038807,-0.006594,-0.067391,-0.078903,-0.042307,0.168062,0.018555,0.032825,0.048606,-0.025697
7734,10.1145_3461702.3462538_54,10.1145_3461702.3462538,798,0,NOT RELEVANT,0.022366,-0.126972,-0.002636,-0.020115,0.033181,...,0.018116,-0.024198,-0.048525,0.055243,-0.060009,0.122534,0.007985,0.048859,0.016293,0.011958


In [15]:
# Save to the requested path
reduced_csv_path = project_root / 'Data' / 'train' / 'train_data_chunks_reduced.csv'
merged_df.to_csv(reduced_csv_path, index=False)
reduced_csv_path, merged_df.shape

(PosixPath('/Users/taishajoseph/Documents/Projects/MDC-Challenge-2025/Data/train/train_data_chunks_reduced.csv'),
 (7736, 46))